In [1]:
# !wget https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions_dedup.json.gz

In [8]:
import pandas as pd

chunksize = 1_000_000  # 100만 행 단위로
chunks = pd.read_json(
    "goodreads_interactions_dedup.json.gz",
    lines=True,
    compression="gzip",
    chunksize=chunksize
)

In [3]:
out_path = "user_book_sequence.csv"
user_seq_dict = {}

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i}...")

    # 필요한 컬럼만 추림
    chunk = chunk[['user_id', 'book_id', 'is_read', 'date_added']]

    # 읽은 책만
    chunk = chunk[chunk['is_read'] == True]

    # 날짜 변환
    chunk['date_added'] = pd.to_datetime(
        chunk['date_added'],
        format="%a %b %d %H:%M:%S %z %Y",
        errors='coerce',
        utc=True
    )
    chunk = chunk.dropna(subset=['date_added'])

    # 유저별로 책 목록 누적
    for uid, group in chunk.groupby('user_id'):
        seq = group.sort_values('date_added')['book_id'].tolist()

        if uid in user_seq_dict:
            user_seq_dict[uid]['book_seq'].extend(seq)
        else:
            user_seq_dict[uid] = {'book_seq': seq}

# dict → DataFrame 변환
records = [
    {'user_id': uid, 'book_seq': seqs['book_seq']}
    for uid, seqs in user_seq_dict.items()
]

df_seq = pd.DataFrame(records)
df_seq.to_parquet(out_path.replace('.csv', '.parquet'), compression='snappy')

print("✅ Done! user_book_sequence.parquet saved.")

Processing chunk 0...
Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing chunk 11...
Processing chunk 12...
Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Processing chunk 17...
Processing chunk 18...
Processing chunk 19...
Processing chunk 20...
Processing chunk 21...
Processing chunk 22...
Processing chunk 23...
Processing chunk 24...
Processing chunk 25...
Processing chunk 26...
Processing chunk 27...
Processing chunk 28...
Processing chunk 29...
Processing chunk 30...
Processing chunk 31...
Processing chunk 32...
Processing chunk 33...
Processing chunk 34...
Processing chunk 35...
Processing chunk 36...
Processing chunk 37...
Processing chunk 38...
Processing chunk 39...
Processing chunk 40...
Processing chunk 41...
Processing chunk 42...
Processing chunk 43..

In [ ]:
롱시퀀스 → “다중 윈도우”로 데이터 늘리기

✅ 하면 데이터 다양성 확 늘어남.

예시:
한 유저가 1000권 읽었으면 → 슬라이딩 윈도우로 여러 training sample 생성

window_size = 100
stride = 50

for i in range(0, len(seq) - window_size, stride):
    subseq = seq[i:i+window_size]
    next_item = seq[i+window_size]

In [9]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

out_path = "user_book_interaction.parquet"
writer = None

# 최종적으로 이런 feature set을 모델에 넣음
# columns = ['user_id', 'book_id', 'rating', 'is_read', 'conflict_flag']


for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i}...")

    chunk = chunk.loc[:, ['user_id', 'book_id', 'rating', 'is_read']].copy()
    # chunk['liked_score'] = (
    #     0.7 * (chunk['rating'] / 5.0).fillna(0) +
    #     0.3 * chunk['is_read'].astype(int)
    # )

    # conflict flag
    chunk['conflict_flag'] = (
        chunk['rating'].notna() & (~chunk['is_read'])
    ).astype(int)
    
    # 결측치/정규화 처리
    chunk['rating'] = chunk['rating'].fillna(0) / 5.0  # [0,1]로 정규화
    chunk['is_read'] = chunk['is_read'].astype(int)

    # Arrow Table로 변환
    table = pa.Table.from_pandas(chunk)

    if writer is None:
        writer = pq.ParquetWriter(out_path, table.schema, compression='snappy')

    writer.write_table(table)

if writer:
    writer.close()

print("✅ Done! user_book_interaction.parquet saved.")

Processing chunk 0...
Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing chunk 11...
Processing chunk 12...
Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Processing chunk 17...
Processing chunk 18...
Processing chunk 19...
Processing chunk 20...
Processing chunk 21...
Processing chunk 22...
Processing chunk 23...
Processing chunk 24...
Processing chunk 25...
Processing chunk 26...
Processing chunk 27...
Processing chunk 28...
Processing chunk 29...
Processing chunk 30...
Processing chunk 31...
Processing chunk 32...
Processing chunk 33...
Processing chunk 34...
Processing chunk 35...
Processing chunk 36...
Processing chunk 37...
Processing chunk 38...
Processing chunk 39...
Processing chunk 40...
Processing chunk 41...
Processing chunk 42...
Processing chunk 43..

In [ ]:
# # 아 안되겠다 걍 책 반납할때쯤 rating을 넣어버리자;;;
# out_path = "user_book_interaction.csv" # for FT-Transformer
# is_first = True

# for chunk in chunks:
#     # 필요한 컬럼만 추림
#     chunk = chunk[['user_id', 'book_id', 'rating', 'is_read']]

#     # liked_score 계산
#     chunk['liked_score'] = (
#         0.7 * (chunk['rating'] / 5.0).fillna(0) +  # 평점 반영
#         0.3 * chunk['is_read'].astype(int)       # 읽었는지 반영
#     )

#     # 저장 (append 모드)
#     chunk.to_csv(out_path, mode='w' if is_first else 'a', index=False, header=is_first)
#     is_first = False

# """
# 순위학습(Ranking, BPR) / pairwise pos-neg / Custom BPR loss
# => "A를 B보다 선호" 관계 학습


# 여기선 negitive sampling도 필수임
# """

In [ ]:
"""
우리 그 시스템에서 테이블 만들려면 한 이정도..?

{
	"bookId": 1,
	"title": "삼국지",
	"author": "이문열",
	"publisher": "미래엔아이세움",
	"category": "역사",
	"isbn": "979-11-362-7519-6",
	"description": "무슨무슨 내용입니다~",
}
=> goodreads_books.json으로 책 정보 불러오고(여기엔 장르 없음)
=> goodreads_book_genres_initial.json으로 장르 불러오자
"""